# Урок 5. Мини-исследование: спортивный аналитик АПЛ

**Имя и фамилия:**  
**Группа:**  
**Номера назначенных команд:**


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Как работать

Структура вашей работы в репозитории:
python-ai-course/
└── homework/
    └── lesson-05/
        └── solution.ipynb

1. Откройте или сохраните этот ноутбук как homework/lesson-05/solution.ipynb.
2. Запускайте ячейки сверху вниз. Если CSV ещё не загружен в текущий сеанс Colab, появится выбор файла. Выберите выданный преподавателем premier_league_2025_26.csv — можно найти его в той же папке на компьютере, где лежит ноутбук. Colab скопирует файл во временную папку /content. После перезапуска сеанса его может понадобиться выбрать снова: /content не является папкой Google Диска и очищается вместе с сеансом.
3. В ячейку с номерами внесите ровно пять назначенных вам номеров через запятую, например [2, 7, 11, 15, 19].
4. В отмеченных Code-ячейках дописывайте Python-код. Первые готовые примеры можно копировать и менять под следующий вопрос.
5. После вычислений запишите вывод под каждой задачей. Подкрепляйте каждое утверждение числами и матчами из результатов ячеек. Не делайте вывод только по просмотру исходной таблицы.
6. Во время занятия сохраните промежуточную версию отдельным коммитом. Дома завершите ноутбук, запустите Restart Kernel → Run All, исправьте ошибки и сохраните итог отдельным коммитом.

Отдельная папка с результатами не создаётся. Справочник команд, ваши назначения и результаты кода показываются под соответствующими ячейками. В репозитории сохраняется сам ноутбук homework/lesson-05/solution.ipynb; загруженный CSV остаётся во временной среде Colab.


## Исследовательская задача

Вы — спортивный аналитик. Исследуйте пять назначенных команд, сравнивайте показатели и обосновывайте выводы данными.

Преподаватель заранее проверил набор: в нём 380 матчей. Искать пропуски и проверять полноту не нужно. Каждая строка — матч. HomeTeam — хозяева, AwayTeam — гости. Например, в Liverpool — Bournemouth со счётом 4:2 команда Liverpool забила 4 гола, а Bournemouth — 2. В таблице это FTHG и FTAG; FTR="H" означает победу хозяев, FTR="A" — победу гостей, D — ничью.

Чтобы сравнивать команды одинаково, подготовленная ячейка создаёт запись с позиции каждой команды: goals_for — её голы, goals_against — голы соперника, *_for и *_against — её показатели и показатели соперника. Для гостя значения уже переведены в его перспективу. Запустите ячейку и используйте её результаты; вручную «переворачивать» матч не нужно.

У каждого этапа есть пример для понимания типа анализа. Это иллюстрация, а не единственный правильный ответ: можно выбрать другой показатель, пример и способ расчёта, если код соответствует вопросу, а вывод подтверждён числами.


| Столбец | Что означает |
|---|---|
| `Date` | дата матча |
| `HomeTeam` | команда хозяев, играет на своём поле |
| `AwayTeam` | команда гостей, играет на поле хозяев |
| `FTHG` | голы хозяев за весь матч |
| `FTAG` | голы гостей за весь матч |
| `FTR` | итог: `H` — победили хозяева, `D` — ничья, `A` — победили гости |
| `HS` / `AS` | все удары хозяев / гостей |
| `HST` / `AST` | удары в створ ворот хозяев / гостей |
| `HC` / `AC` | угловые хозяев / гостей |
| `HF` / `AF` | фолы хозяев / гостей |
| `HY` / `AY` | жёлтые карточки хозяев / гостей |
| `HR` / `AR` | красные карточки хозяев / гостей |

## Загрузка набора данных

Если premier_league_2025_26.csv уже находится в текущей среде Colab (/content), ячейка прочитает его сразу. Иначе откроется выбор файла: выберите CSV с компьютера. Даже если файл лежит рядом с ноутбуком в Google Диске, Colab не помещает его автоматически в /content; его нужно выбрать для текущего сеанса.

После загрузки ячейка покажет число матчей и команд. Список команд с номерами появится в выводе ячейки «Этап 1» — никакая отдельная папка или файл для списка не создаётся.


In [20]:
from pathlib import Path
import csv

file_name = "premier_league_2025_26.csv"
data_path = Path("/content") / file_name

if not data_path.exists():
    from google.colab import files
    print("Выберите premier_league_2025_26.csv с компьютера. Файл будет помещён во временную среду Colab /content.")
    uploaded_files = files.upload()
    if file_name not in uploaded_files:
        raise FileNotFoundError("Выберите файл premier_league_2025_26.csv")

numeric_fields = ["FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF", "HY", "AY", "HR", "AR"]
with data_path.open(encoding="utf-8-sig", newline="") as f:
    matches = list(csv.DictReader(f))
for match in matches:
    for field in numeric_fields:
        match[field] = int(match[field])
all_teams = sorted({m["HomeTeam"] for m in matches} | {m["AwayTeam"] for m in matches}, key=str.casefold)
team_catalog = {n: name for n, name in enumerate(all_teams, start=1)}
print(f"Загружено матчей: {len(matches)}; команд: {len(team_catalog)}")


Загружено матчей: 380; команд: 20


## Этап 1. Справочник команд и ваши номера

После загрузки файла ноутбук собирает все уникальные названия из столбцов хозяев и гостей, сортирует их по алфавиту и присваивает номера с 1. Это справочник `team_catalog`: номер → название команды. Он создаётся в памяти текущего сеанса Colab, а не сохраняется в отдельной папке или файле.

Справочник будет показан в выводе ячейки ниже. Преподаватель сообщит вам пять назначенных номеров. Впишите их **через запятую**, например `[2, 7, 11, 15, 19]`. После запуска ячейка покажет названия ваших команд; они будут доступны ноутбуку в списке `selected_team_names`.

In [21]:
assigned_team_numbers = [ 4, 6, 10, 11, 18]  # Впишите свои пять номеров через запятую, например [2, 7, 11, 15, 19].
selected_team_names = []
if len(assigned_team_numbers)==5 and len(set(assigned_team_numbers))==5 and all(n in team_catalog for n in assigned_team_numbers):
    selected_team_names=[team_catalog[n] for n in assigned_team_numbers]
    print("Ваши команды:", selected_team_names)
else:
    print("Впишите ровно пять разных номеров через запятую.")
print("Справочник команд — номер и название:")
for number,name in team_catalog.items():
    print(f"{number:>2}. {name}")

Ваши команды: ['Brentford', 'Burnley', 'Fulham', 'Leeds', 'Tottenham']
Справочник команд — номер и название:
 1. Arsenal
 2. Aston Villa
 3. Bournemouth
 4. Brentford
 5. Brighton
 6. Burnley
 7. Chelsea
 8. Crystal Palace
 9. Everton
10. Fulham
11. Leeds
12. Liverpool
13. Man City
14. Man United
15. Newcastle
16. Nott'm Forest
17. Sunderland
18. Tottenham
19. West Ham
20. Wolves


## Этап 2. Подготовленные записи матчей

Хозяева играют на своём поле и записаны первыми; гости — на поле хозяев и записаны вторыми. Например, в матче Liverpool — Bournemouth 4:2 у Liverpool забитые голы — 4, у Bournemouth — 2.

Следующая ячейка уже готова: она для всех 380 матчей создаёт записи обеих команд. В каждой записи показатели собраны одинаково: `goals_for` — голы этой команды, `goals_against` — голы соперника; `*_for` и `*_against` так же обозначают её показатели и показатели соперника. Результат `W/D/L` определяется по счёту. Вам не нужно писать или менять эту ячейку: запустите её и дальше анализируйте `selected_team_games` — матчи пяти назначенных команд.

In [22]:
team_games = []

for match in matches:
    sides = [
        {
            "team": match["HomeTeam"], "opponent": match["AwayTeam"], "venue": "дома",
            "goals_for": match["FTHG"], "goals_against": match["FTAG"],
            "shots_for": match["HS"], "shots_against": match["AS"],
            "target_for": match["HST"], "target_against": match["AST"],
            "corners_for": match["HC"], "corners_against": match["AC"],
            "fouls_for": match["HF"], "fouls_against": match["AF"],
            "yellow_for": match["HY"], "yellow_against": match["AY"],
            "red_for": match["HR"], "red_against": match["AR"],
            "win_code": "H"
        },
        {
            "team": match["AwayTeam"], "opponent": match["HomeTeam"], "venue": "в гостях",
            "goals_for": match["FTAG"], "goals_against": match["FTHG"],
            "shots_for": match["AS"], "shots_against": match["HS"],
            "target_for": match["AST"], "target_against": match["HST"],
            "corners_for": match["AC"], "corners_against": match["HC"],
            "fouls_for": match["AF"], "fouls_against": match["HF"],
            "yellow_for": match["AY"], "yellow_against": match["HY"],
            "red_for": match["AR"], "red_against": match["HR"],
            "win_code": "A"
        }
    ]

    for side in sides:
        if side["goals_for"] > side["goals_against"]:
            result = "W"
        elif side["goals_for"] == side["goals_against"]:
            result = "D"
        else:
            result = "L"
        side["date"] = match["Date"]
        side["result"] = result
        side["points"] = 3 if result == "W" else 1 if result == "D" else 0
        del side["win_code"]
        team_games.append(side)

selected_team_games = []
for game in team_games:
    if game["team"] in selected_team_names:
        selected_team_games.append(game)

print("Записей по всем командам:", len(team_games))
print("Записей по вашим пяти командам:", len(selected_team_games))
for name in selected_team_names:
    games_count = 0
    for game in selected_team_games:
        if game["team"] == name:
            games_count += 1
    print(name, "матчей:", games_count)

Записей по всем командам: 760
Записей по вашим пяти командам: 190
Brentford матчей: 38
Burnley матчей: 38
Fulham матчей: 38
Leeds матчей: 38
Tottenham матчей: 38


## Этап 3. Профиль сезона

**Вопросы 1–2.** Для каждой из пяти команд посчитайте победы, ничьи, поражения, забитые и пропущенные голы, разницу голов и очки (победа — 3, ничья — 1, поражение — 0). Кто лидер среди ваших команд по очкам? Совпадает ли он с лидером по разнице голов?

Ниже дан готовый пример сводки. Запустите его и посмотрите, как список матчей превращается в итоги сезона. В следующих этапах можно скопировать цикл и изменить показатели.

**Пример формата вывода:** в этом наборе Arsenal закончил сезон с 85 очками и разницей голов +44, а Man City — с 78 очками и разницей +42. Для своей пятёрки сравните рассчитанные строки. Это пример чтения сводки; возможны другие способы расчёта и оформления.


In [23]:
team_profiles = []

for team in selected_team_names:
    wins = 0
    draws = 0
    losses = 0
    goals_for = 0
    goals_against = 0

    for game in selected_team_games:
        if game["team"] == team:
            goals_for += game["goals_for"]
            goals_against += game["goals_against"]
            if game["result"] == "W":
                wins += 1
            elif game["result"] == "D":
                draws += 1
            else:
                losses += 1

    goal_difference = goals_for - goals_against
    points = wins * 3 + draws
    team_profiles.append({
        "team": team, "wins": wins, "draws": draws, "losses": losses,
        "goals_for": goals_for, "goals_against": goals_against,
        "goal_difference": goal_difference, "points": points
    })

for profile in team_profiles:
    print(profile)

{'team': 'Brentford', 'wins': 14, 'draws': 11, 'losses': 13, 'goals_for': 55, 'goals_against': 52, 'goal_difference': 3, 'points': 53}
{'team': 'Burnley', 'wins': 4, 'draws': 10, 'losses': 24, 'goals_for': 38, 'goals_against': 75, 'goal_difference': -37, 'points': 22}
{'team': 'Fulham', 'wins': 15, 'draws': 7, 'losses': 16, 'goals_for': 47, 'goals_against': 51, 'goal_difference': -4, 'points': 52}
{'team': 'Leeds', 'wins': 11, 'draws': 14, 'losses': 13, 'goals_for': 49, 'goals_against': 56, 'goal_difference': -7, 'points': 47}
{'team': 'Tottenham', 'wins': 10, 'draws': 11, 'losses': 17, 'goals_for': 48, 'goals_against': 57, 'goal_difference': -9, 'points': 41}


По очкам лидирует [команда] — [X] очков (W–D–L: [a–b–c]). По разнице голов лидер — [команда] с показателем [+Y]. Лидеры совпадают / не совпадают: [проверь по выводу team_profiles]. Например, у [команда] разница [+Y], а у [команда] — [+Z], хотя очков у них [поровну / разное число].### ? ВЫВОДЫ

Назовите лидера по очкам и по разнице голов и подтвердите числами.

**Ответ:**

## Этап 4. Домашние и выездные матчи

**Вопрос 3.** Для каждой команды сравните число побед и среднее количество забитых голов дома и в гостях. Для каждого среднего укажите число матчей, на котором оно рассчитано.

**Пример:** в наборе Man City забивал в среднем 2,37 гола дома и 1,68 в гостях (по 19 матчей). Покажите такой же расчёт для своих команд. Среднее можно получить суммой голов, поделённой на число матчей; допустим и другой понятный способ.


In [24]:
home_away_summary = []

for team in selected_team_names:
    home_games = [g for g in selected_team_games if g["team"] == team and g["venue"] == "дома"]
    away_games = [g for g in selected_team_games if g["team"] == team and g["venue"] == "в гостях"]

    home_wins = sum(1 for g in home_games if g["result"] == "W")
    away_wins = sum(1 for g in away_games if g["result"] == "W")
    home_goals = sum(g["goals_for"] for g in home_games)
    away_goals = sum(g["goals_for"] for g in away_games)

    home_avg = home_goals / len(home_games) if home_games else 0
    away_avg = away_goals / len(away_games) if away_games else 0

    home_away_summary.append({
        "team": team,
        "home_matches": len(home_games),
        "home_wins": home_wins,
        "home_goals_avg": round(home_avg, 2),
        "away_matches": len(away_games),
        "away_wins": away_wins,
        "away_goals_avg": round(away_avg, 2),
        "diff_avg": round(home_avg - away_avg, 2),
    })

for summary in home_away_summary:
    print(summary)

{'team': 'Brentford', 'home_matches': 19, 'home_wins': 8, 'home_goals_avg': 1.74, 'away_matches': 19, 'away_wins': 6, 'away_goals_avg': 1.16, 'diff_avg': 0.58}
{'team': 'Burnley', 'home_matches': 19, 'home_wins': 2, 'home_goals_avg': 0.95, 'away_matches': 19, 'away_wins': 2, 'away_goals_avg': 1.05, 'diff_avg': -0.11}
{'team': 'Fulham', 'home_matches': 19, 'home_wins': 11, 'home_goals_avg': 1.58, 'away_matches': 19, 'away_wins': 4, 'away_goals_avg': 0.89, 'diff_avg': 0.68}
{'team': 'Leeds', 'home_matches': 19, 'home_wins': 9, 'home_goals_avg': 1.53, 'away_matches': 19, 'away_wins': 2, 'away_goals_avg': 1.05, 'diff_avg': 0.47}
{'team': 'Tottenham', 'home_matches': 19, 'home_wins': 3, 'home_goals_avg': 1.16, 'away_matches': 19, 'away_wins': 7, 'away_goals_avg': 1.37, 'diff_avg': -0.21}


### ? ВЫВОДЫ

У какой команды заметнее разница между домашней и выездной игрой? Приведите числа.

**Ответ:** Самая заметная разница дом/гости у [команда]: дома — [X] голов в среднем за [N] матчей и [W] побед, в гостях — [Y] голов за [M] матчей и [V] побед; разница средних — [D]. Для сравнения, у [другая команда] дома [X2] за [N2] матчей, в гостях [Y2] за [M2] — разница всего [D2].

## Этап 5. Проверка гипотезы

**Вопросы 4–5.** Сравните выбранные команды по голам и ударам в створ. Проверьте гипотезу: «Преимущество по ударам в створ связано с победой». Для каждой команды посчитайте матчи, где она нанесла больше ударов в створ, число побед в них и отдельно матчи с равным числом ударов в створ.

**Пример контрпримера:** 24.11.2025 Everton победил Man United 1:0, хотя уступил по ударам в створ 1:6 и по общему числу ударов 3:25. Этот матч показывает, почему связь не означает, что преимущество гарантирует победу. Можно привести другой матч из результатов своей ячейки.


In [25]:

target_comparison = []

for team in selected_team_names:
    games = [g for g in selected_team_games if g["team"] == team]
    more_target = [g for g in games if g["target_for"] > g["target_against"]]
    more_target_wins = [g for g in more_target if g["result"] == "W"]
    equal_target = [g for g in games if g["target_for"] == g["target_against"]]

    target_comparison.append({
        "team": team,
        "matches": len(games),
        "more_target_matches": len(more_target),
        "more_target_wins": len(more_target_wins),
        "more_target_win_rate": round(len(more_target_wins) / len(more_target), 2) if more_target else 0,
        "equal_target_matches": len(equal_target),
    })

for result in target_comparison:
    print(result)

{'team': 'Brentford', 'matches': 38, 'more_target_matches': 15, 'more_target_wins': 11, 'more_target_win_rate': 0.73, 'equal_target_matches': 4}
{'team': 'Burnley', 'matches': 38, 'more_target_matches': 7, 'more_target_wins': 1, 'more_target_win_rate': 0.14, 'equal_target_matches': 3}
{'team': 'Fulham', 'matches': 38, 'more_target_matches': 14, 'more_target_wins': 9, 'more_target_win_rate': 0.64, 'equal_target_matches': 6}
{'team': 'Leeds', 'matches': 38, 'more_target_matches': 19, 'more_target_wins': 7, 'more_target_win_rate': 0.37, 'equal_target_matches': 2}
{'team': 'Tottenham', 'matches': 38, 'more_target_matches': 19, 'more_target_wins': 8, 'more_target_win_rate': 0.42, 'equal_target_matches': 4}


### ? ВЫВОДЫ

Поддерживают ли результаты гипотезу? Приведите пример за и контрпример.

**Ответ:**  Гипотеза в среднем поддерживается: у команд, где ударов в створ больше, доля побед высокая — например, у [команда] в [K] таких матчах [P] побед (win_rate [R]). Но связь не абсолютна — есть контрпример: [дата] [команда] победила [соперник] со счётом [X:Y], хотя уступила по ударам в створ [A:B]. Значит, преимущество по ударам в створ повышает шансы, но не гарантирует победу.

## Этап 6. Матч-детектив

**Вопрос 6.** Найдите победу при меньшем числе ударов в створ или поражение при большем. Укажите дату, команды, счёт и удары в створ с обеих сторон.

**Вопрос 7.** У какой из ваших пяти команд была самая крупная победа? Найдите матч, укажите дату, соперников и счёт, затем посчитайте разницу забитых и пропущенных голов. Если у нескольких матчей одинаковый максимум, приведите их все.

**Пример по всему набору:** максимальная разница в счёте — 5 голов. Например, 23.08.2025 Arsenal победил Leeds 5:0; ещё один матч с такой разницей — Sunderland — Nott'm Forest 0:5 (24.04.2026). Для ответа используйте только матчи ваших назначенных команд.

**На выбор — выполните один дополнительный пункт:**
- Найдите матч одной из ваших команд, в котором забили больше всего голов суммарно. Укажите счёт и сравните его с обычным результатом команды.
- Какая из ваших команд чаще всего не пропускала? Посчитайте такие матчи и приведите пример.

Пример для дополнительного варианта с максимумом голов: 02.12.2025 Fulham — Man City 4:5, всего 9 голов. Для «сухих» матчей, например, можно найти среди результатов Arsenal — Leeds 5:0 (23.08.2025). Это только иллюстрации; допустим другой пример и другой способ поиска, если код и вывод отвечают на вопрос.


In [26]:
surprising_matches = []

for g in selected_team_games:
    # победа при меньшем числе ударов в створ или поражение при большем
    if (g["result"] == "W" and g["target_for"] < g["target_against"]) or \
       (g["result"] == "L" and g["target_for"] > g["target_against"]):
        surprising_matches.append({
            "date": g["date"],
            "team": g["team"],
            "opponent": g["opponent"],
            "venue": g["venue"],
            "score": f'{g["goals_for"]}:{g["goals_against"]}',
            "targets": f'{g["target_for"]}:{g["target_against"]}',
            "result": g["result"],
        })

for match in surprising_matches[:10]:
    print(match)

{'date': '30/08/2025', 'team': 'Brentford', 'opponent': 'Sunderland', 'venue': 'в гостях', 'score': '1:2', 'targets': '4:3', 'result': 'L'}
{'date': '20/09/2025', 'team': 'Leeds', 'opponent': 'Wolves', 'venue': 'в гостях', 'score': '3:1', 'targets': '4:6', 'result': 'W'}
{'date': '04/10/2025', 'team': 'Leeds', 'opponent': 'Tottenham', 'venue': 'дома', 'score': '1:2', 'targets': '4:3', 'result': 'L'}
{'date': '04/10/2025', 'team': 'Tottenham', 'opponent': 'Leeds', 'venue': 'в гостях', 'score': '2:1', 'targets': '3:4', 'result': 'W'}
{'date': '18/10/2025', 'team': 'Burnley', 'opponent': 'Leeds', 'venue': 'дома', 'score': '2:0', 'targets': '3:4', 'result': 'W'}
{'date': '18/10/2025', 'team': 'Leeds', 'opponent': 'Burnley', 'venue': 'в гостях', 'score': '0:2', 'targets': '4:3', 'result': 'L'}
{'date': '19/10/2025', 'team': 'Tottenham', 'opponent': 'Aston Villa', 'venue': 'дома', 'score': '1:2', 'targets': '3:2', 'result': 'L'}
{'date': '08/11/2025', 'team': 'Burnley', 'opponent': 'West Ham

In [27]:
largest_wins = []
max_margin = -1

for game in selected_team_games:
    if game["result"] == "W":
        margin = game["goals_for"] - game["goals_against"]
        if margin > max_margin:
            max_margin = margin
            largest_wins = [game]
        elif margin == max_margin:
            largest_wins.append(game)

print("Максимальная разница:", max_margin)
for g in largest_wins:
    print(g["date"], g["team"], "—", g["opponent"], f'{g["goals_for"]}:{g["goals_against"]}')

# Доп. вариант 1: матчи с максимумом голов
high_scoring_matches = []
max_total = -1
for game in selected_team_games:
    total = game["goals_for"] + game["goals_against"]
    if total > max_total:
        max_total = total
        high_scoring_matches = [game]
    elif total == max_total:
        high_scoring_matches.append(game)

print("\nМаксимум голов в матче:", max_total)
for g in high_scoring_matches:
    print(g["date"], g["team"], "—", g["opponent"], f'{g["goals_for"]}:{g["goals_against"]}')

# Доп. вариант 2: сухие матчи
clean_sheet_counts = []
for team in selected_team_names:
    clean = [g for g in selected_team_games if g["team"] == team and g["goals_against"] == 0]
    clean_sheet_counts.append({"team": team, "clean_sheets": len(clean)})

print("\nМатчи без пропущенных голов:")
for item in clean_sheet_counts:
    print(item)


Максимальная разница: 3
16/08/2025 Tottenham — Burnley 3:0
13/09/2025 Tottenham — West Ham 3:0
26/10/2025 Tottenham — Everton 3:0
01/11/2025 Fulham — Wolves 3:0
20/12/2025 Leeds — Crystal Palace 4:1
27/12/2025 Brentford — Bournemouth 4:1
07/01/2026 Brentford — Sunderland 3:0
18/04/2026 Leeds — Wolves 3:0
02/05/2026 Brentford — West Ham 3:0

Максимум голов в матче: 9
02/12/2025 Fulham — Man City 4:5

Матчи без пропущенных голов:
{'team': 'Brentford', 'clean_sheets': 10}
{'team': 'Burnley', 'clean_sheets': 4}
{'team': 'Fulham', 'clean_sheets': 9}
{'team': 'Leeds', 'clean_sheets': 8}
{'team': 'Tottenham', 'clean_sheets': 9}


### ? ВЫВОДЫ

Объясните, что показывает найденный матч и чего он не доказывает. Для победы с максимальной разницей приведите счёт и расчёт.

**Ответ:** По очкам лидирует [команда] — [X] очков (W–D–L: [a–b–c]). По разнице голов лидер — [команда] с показателем [+Y]. Лидеры совпадают / не совпадают: [проверь по выводу team_profiles]. Например, у [команда] разница [+Y], а у [команда] — [+Z], хотя очков у них [поровну / разное число].


## Этап 7. Угловые и карточки

**Вопросы 8–9.** У какой из ваших команд больше угловых в среднем? Сопоставьте угловые с очками или победами. Посчитайте жёлтые и красные карточки отдельно. Всегда ли команда с большим числом карточек проигрывала?

**Пример неожиданного матча:** Контрпример: [дата] [команда] — [соперник], счёт [X:Y], удары в створ [A:B] — победа при меньшем числе ударов в створ (или поражение при большем). Это показывает, что реализация моментов важнее их количества, но один матч не доказывает закономерность.

Самая крупная победа: [дата] [команда] — [соперник] со счётом [X:Y], разница голов [N]. [Если несколько — перечисли все.]

Доп. вариант: больше всего голов в матче — [дата] [команда] — [соперник], [X:Y], всего [T] голов, что выше обычного среднего команды [S] голов за матч. / Либо: чаще всех не пропускала [команда] — [N] «сухих» матчей, например [дата] против [соперник] [X:0].Контрпример: [дата] [команда] — [соперник], счёт [X:Y], удары в створ [A:B] — победа при меньшем числе ударов в створ (или поражение при большем). Это показывает, что реализация моментов важнее их количества, но один матч не доказывает закономерность.

Самая крупная победа: [дата] [команда] — [соперник] со счётом [X:Y], разница голов [N]. [Если несколько — перечисли все.]

Доп. вариант: больше всего голов в матче — [дата] [команда] — [соперник], [X:Y], всего [T] голов, что выше обычного среднего команды [S] голов за матч. / Либо: чаще всех не пропускала [команда] — [N] «сухих» матчей, например [дата] против [соперник] [X:0].


In [29]:
corners_cards_summary = []

for team in selected_team_names:
    games = [g for g in selected_team_games if g["team"] == team]
    n = len(games)
    corners_avg = sum(g["corners_for"] for g in games) / n
    yellow_avg = sum(g["yellow_for"] for g in games) / n
    red_total = sum(g["red_for"] for g in games)
    wins = sum(1 for g in games if g["result"] == "W")

    corners_cards_summary.append({
        "team": team,
        "matches": n,
        "corners_avg": round(corners_avg, 2),
        "yellow_avg": round(yellow_avg, 2),
        "red_total": red_total,
        "wins": wins,
    })

for summary in corners_cards_summary:
    print(summary)

{'team': 'Brentford', 'matches': 38, 'corners_avg': 4.79, 'yellow_avg': 1.82, 'red_total': 0, 'wins': 14}
{'team': 'Burnley', 'matches': 38, 'corners_avg': 3.63, 'yellow_avg': 1.68, 'red_total': 2, 'wins': 4}
{'team': 'Fulham', 'matches': 38, 'corners_avg': 4.95, 'yellow_avg': 1.97, 'red_total': 1, 'wins': 15}
{'team': 'Leeds', 'matches': 38, 'corners_avg': 4.42, 'yellow_avg': 1.63, 'red_total': 1, 'wins': 11}
{'team': 'Tottenham', 'matches': 38, 'corners_avg': 5.47, 'yellow_avg': 2.66, 'red_total': 3, 'wins': 10}


### ? ВЫВОДЫ

Приведите один пример за или против связи показателя с результатом. Не утверждайте причинность.

**Ответ:** Больше всего угловых в среднем у [команда] — [X] за матч, но это не гарантирует очков: например, [дата] [команда] подала [A] угловых против [B], а результат — [счёт]. По карточкам: больше всех жёлтых в среднем у [команда] ([X]), красных всего [N] у [команда]. Команда с большим числом карточек не всегда проигрывала — пример: [дата] [команда] получила [A] жёлтых, но [победила/сыграла вничью] [счёт]. Причинность по этим данным утверждать нельзя.


## Этап 8. Сравнение и вывод аналитика

**Вопросы 10–13.** Выберите две из своих пяти команд с похожим числом очков или побед и сравните их по двум другим показателям. Сформулируйте три вывода по всем пяти командам; каждый подтвердите результатом кода. Добавьте один неожиданный результат и вопрос, на который этот набор данных не отвечает.

**Пример выбора:** можно сравнить команды с близкими очками, но разным числом забитых голов или разницей голов. Например, Arsenal (85 очков) и Man City (78) — близкие лидеры сезона, но этот пример не подменяет выбор команд из вашего задания. Подойдут и другие команды с другими показателями.


In [30]:
compare_team_numbers = [4, 18]  # Aston Villa и Liverpool — поменяй при желании
compare_names = [team_catalog[n] for n in compare_team_numbers]

comparison = [p for p in team_profiles if p["team"] in compare_names]
for item in comparison:
    print(item)

# дополнительные показатели для сравнения
for name in compare_names:
    games = [g for g in selected_team_games if g["team"] == name]
    shots_avg = round(sum(g["shots_for"] for g in games) / len(games), 2)
    target_avg = round(sum(g["target_for"] for g in games) / len(games), 2)
    corners_avg = round(sum(g["corners_for"] for g in games) / len(games), 2)
    print(f"{name}: удары/матч {shots_avg}, в створ/матч {target_avg}, угловые/матч {corners_avg}")

{'team': 'Brentford', 'wins': 14, 'draws': 11, 'losses': 13, 'goals_for': 55, 'goals_against': 52, 'goal_difference': 3, 'points': 53}
{'team': 'Tottenham', 'wins': 10, 'draws': 11, 'losses': 17, 'goals_for': 48, 'goals_against': 57, 'goal_difference': -9, 'points': 41}
Brentford: удары/матч 10.66, в створ/матч 3.87, угловые/матч 4.79
Tottenham: удары/матч 11.47, в створ/матч 3.95, угловые/матч 5.47


### ? ВЫВОДЫ — итог мини-исследования

**Три вывода с доказательствами:**

1. Лидер по очкам и разнице голов — [команда]: [X] очков и разница [+Y] (W–D–L: [a–b–c]). У ближайшей [команда] — [X2] очков и [+Y2], то есть отрыв [D] очков.  
2.  Домашний фактор заметен у [команда]: дома [X] голов за матч и [W] побед, в гостях [Y] голов и [V] побед — разница средних [D]. У [другая команда] разница всего [D2], поэтому преимущество своего поля у неё слабее.
3.  Преимущество по ударам в створ связано с победами, но не гарантирует их: у [команда] win_rate в таких матчах [R] ([P] из [K]), однако [дата] [команда] победила [соперник] [X:Y] при ударах в створ [A:B].Преимущество по ударам в створ связано с победами, но не гарантирует их: у [команда] win_rate в таких матчах [R] ([P] из [K]), однако [дата] [команда] победила [соперник] [X:Y] при ударах в створ [A:B].

**Неожиданный матч или результат:**  [дата] [команда] — [соперник], счёт [X:Y] при ударах в створ [A:B] — победа вопреки статистике ударов.



**Вопрос, на который данных не хватает:**
«Почему команда с преимуществом по ударам в створ проигрывает?» В наборе нет xG, качества моментов, позиций игроков и данных о вратаре, поэтому причину по этим данным установить нельзя.

## Коммиты и домашнее завершение

На занятии сохраните ноутбук homework/lesson-05/solution.ipynb отдельным коммитом с сообщением:

**HW-05 — работа на занятии**

Дома завершите ноутбук в том же файле, выполните Restart Kernel → Run All и проверьте код и текстовые выводы. Сохраните итоговую версию отдельным коммитом:

**HW-05 — итоговая версия**

**Срок: 12 октября 2026 года, 23:59 по Москве.** Не удаляйте промежуточный коммит из истории.
